# OpenCV — Computer Vision from Pixels to Products

## What Is This Notebook About?

Your eyes capture light and your brain instantly understands it: "that's a stop sign, that person is running, that car is too close." **OpenCV (Open Source Computer Vision Library)** gives computers the same ability — loading images, understanding their content, detecting objects, and processing video in real time.

OpenCV is used in **self-driving cars, security cameras, medical imaging, AR filters, and robotics**. If you've used a Snapchat filter or watched a sports broadcast with player tracking, you've seen OpenCV in action.

---

## Why Should You Care?

| Industry | OpenCV Application |
|---|---|
| Autonomous vehicles | Lane detection, pedestrian tracking |
| Healthcare | Cell counting, tumor boundary detection |
| Manufacturing | Defect detection on assembly lines |
| Security | Face recognition, license plate reading |
| Retail | Checkout-free stores (Amazon Go) |
| Sports | Player tracking, ball trajectory analysis |

---

## Prerequisites

- Basic Python (NumPy arrays, loops)
- High school math (coordinates: (x, y) on a grid)

**You do NOT need**: prior image processing knowledge

---

## Table of Contents

1. [Setup & How Images Are Stored](#1-setup)
2. [Reading, Displaying & Writing Images](#2-io)
3. [Color Spaces — BGR, RGB, HSV, Grayscale](#3-color)
4. [Drawing Shapes & Text on Images](#4-drawing)
5. [Image Transformations](#5-transforms)
6. [Filtering & Blurring](#6-filters)
7. [Edge Detection — Canny Algorithm](#7-edges)
8. [Contours — Finding Object Boundaries](#8-contours)
9. [Morphological Operations](#9-morphology)
10. [Histograms & Histogram Equalization](#10-histograms)
11. [Template Matching](#11-template)
12. [Video Processing Basics](#12-video)
13. [Mini Project — Coin Counter](#13-mini-project)
14. [Common Pitfalls](#14-pitfalls)
15. [Interview Q&A](#15-interview)
16. [Resources](#16-resources)
17. [Summary](#17-summary)

---
## 1. Setup & How Images Are Stored <a id='1-setup'></a>

### The LEGO Brick Analogy

An image is like a giant grid of LEGO bricks. Each brick (pixel) has a color made of 3 values: how much **Blue**, **Green**, and **Red** light it emits (0–255 each).

- A 1920×1080 image = 1920 columns × 1080 rows × 3 color channels = **~6.2 million values**
- In NumPy: shape is `(height, width, channels)` = `(1080, 1920, 3)`

**CRITICAL: OpenCV uses BGR (Blue-Green-Red), not RGB (Red-Green-Blue)!**  
This is a historical quirk that trips up every beginner. Always remember: OpenCV = BGR.

In [ ]:
# Install: pip install opencv-python matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version:  {np.__version__}")

# ── What does an image look like as a NumPy array? ────────────────────────────
# Create a tiny 4×4 image manually (just to see the structure)
tiny = np.zeros((4, 4, 3), dtype=np.uint8)   # black image: all zeros
tiny[0, 0] = [255, 0, 0]   # top-left pixel: BLUE (remember BGR!)
tiny[0, 3] = [0, 255, 0]   # top-right pixel: GREEN
tiny[3, 0] = [0, 0, 255]   # bottom-left pixel: RED
tiny[3, 3] = [255, 255, 255]  # bottom-right: WHITE

print("\n=== Image as NumPy Array ===")
print(f"Shape:  {tiny.shape}   (height=4, width=4, channels=3)")
print(f"Dtype:  {tiny.dtype}   (values: 0–255)")
print(f"Size:   {tiny.nbytes} bytes total")
print("\nTop-left pixel (BGR):", tiny[0, 0],  "→ pure Blue")
print("Top-right pixel (BGR):", tiny[0, 3], "→ pure Green")
print("Bottom-left pixel (BGR):", tiny[3, 0], "→ pure Red")
print("Bottom-right pixel (BGR):", tiny[3, 3], "→ White")
print("\nFull array (each row = one row of pixels, each element = [B,G,R]):")
print(tiny)

In [ ]:
# ── Helper: display OpenCV images in Jupyter (cv2.imshow won't work here) ────

def show(img, title='', figsize=(6, 4), cmap=None):
    """Display a BGR or grayscale OpenCV image in a Jupyter notebook."""
    fig, ax = plt.subplots(figsize=figsize)
    if len(img.shape) == 3:
        # OpenCV stores BGR → convert to RGB for matplotlib
        display_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(display_img)
    else:
        ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

def show_multi(images, titles, figsize=None):
    """Display multiple images side by side."""
    n = len(images)
    if figsize is None:
        figsize = (5 * n, 4)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("Helper functions defined!")
print("NOTE: In notebooks we use matplotlib to display images.")
print("      In scripts/apps, cv2.imshow() + cv2.waitKey(0) is used instead.")

---
## 2. Reading, Displaying & Writing Images <a id='2-io'></a>

In [ ]:
# ── Create a synthetic test image (so no file needed) ────────────────────────

# We'll create a 300×400 image with colored blocks as our test image
height, width = 300, 400
test_img = np.zeros((height, width, 3), dtype=np.uint8)

# Fill quadrants with different colors (BGR format!)
test_img[:150, :200]   = [255,   0,   0]   # Top-left: Blue
test_img[:150, 200:]   = [  0, 255,   0]   # Top-right: Green
test_img[150:, :200]   = [  0,   0, 255]   # Bottom-left: Red
test_img[150:, 200:]   = [255, 255,   0]   # Bottom-right: Cyan

# Add a white circle in the center
cv2.circle(test_img, (200, 150), 60, (255, 255, 255), -1)

print(f"Test image created: shape={test_img.shape}, dtype={test_img.dtype}")
show(test_img, 'Test image (synthetic, 400×300 px)')

# ── Save and reload ───────────────────────────────────────────────────────────
cv2.imwrite('/tmp/test_image.jpg', test_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
cv2.imwrite('/tmp/test_image.png', test_img)  # PNG: lossless

reloaded = cv2.imread('/tmp/test_image.png')  # returns BGR np.uint8
print(f"\nReloaded from disk: shape={reloaded.shape}, dtype={reloaded.dtype}")
print("cv2.imread() flags:")
print("  cv2.IMREAD_COLOR     (1) — BGR, 3 channels (default)")
print("  cv2.IMREAD_GRAYSCALE (0) — 1 channel")
print("  cv2.IMREAD_UNCHANGED (-1) — as-is (keeps alpha channel if present)")

gray_reload = cv2.imread('/tmp/test_image.png', cv2.IMREAD_GRAYSCALE)
print(f"\nLoaded as grayscale: shape={gray_reload.shape}")

---
## 3. Color Spaces — BGR, RGB, HSV, Grayscale <a id='3-color'></a>

### The Paint Analogy

Different color *spaces* are like different ways to describe paint:
- **BGR/RGB**: how much blue, green, red paint to mix
- **HSV** (Hue-Saturation-Value): what *color* (hue), how *vivid* (saturation), how *bright* (value)
- **Grayscale**: just brightness, no color
- **LAB**: perceptually uniform (equal numerical distance = equal perceived difference)

**HSV is gold for color detection!** Want to detect red objects? In RGB it's complicated (high R, low G, low B — but lighting changes everything). In HSV, red is just hue ≈ 0° or 360° regardless of lighting.

In [ ]:
# ── Color space conversions ───────────────────────────────────────────────────

img = test_img.copy()

# Convert to various color spaces
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Top row: full images
axes[0,0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0,0].set_title('Original (BGR→RGB)'); axes[0,0].axis('off')
axes[0,1].imshow(gray, cmap='gray');                    axes[0,1].set_title('Grayscale');          axes[0,1].axis('off')
axes[0,2].imshow(hsv);                                  axes[0,2].set_title('HSV (raw values)');   axes[0,2].axis('off')
axes[0,3].imshow(lab);                                  axes[0,3].set_title('LAB color space');    axes[0,3].axis('off')

# Bottom row: individual HSV channels
h, s, v = cv2.split(hsv)
axes[1,0].imshow(h, cmap='hsv');    axes[1,0].set_title('H: Hue (0-179)');         axes[1,0].axis('off')
axes[1,1].imshow(s, cmap='gray');   axes[1,1].set_title('S: Saturation (0-255)');  axes[1,1].axis('off')
axes[1,2].imshow(v, cmap='gray');   axes[1,2].set_title('V: Value/Brightness');    axes[1,2].axis('off')

b, g, r = cv2.split(img)
axes[1,3].imshow(np.stack([r, g, b], axis=2)); axes[1,3].set_title('RGB channels split'); axes[1,3].axis('off')

plt.suptitle('Color Space Comparison', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("\nOpenCV HSV ranges:")
print("  H (Hue):        0–179  (half of 360°)")
print("  S (Saturation): 0–255")
print("  V (Value):      0–255")
print("\nMATLAB/PIL/Pillow use H: 0–360 — be careful when converting ranges!")

In [ ]:
# ── Color masking: detect a specific color using HSV ─────────────────────────

# Create an image with colored circles
color_img = np.zeros((200, 600, 3), dtype=np.uint8)
# Colors in BGR
cv2.circle(color_img, (100, 100), 70, (0,   0, 255), -1)   # Red
cv2.circle(color_img, (300, 100), 70, (0, 255,   0), -1)   # Green
cv2.circle(color_img, (500, 100), 70, (255, 0,   0), -1)   # Blue

hsv_color = cv2.cvtColor(color_img, cv2.COLOR_BGR2HSV)

# Define HSV range for RED (wraps around 0/180 in HSV)
lower_red1 = np.array([0, 120, 70])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 120, 70])
upper_red2 = np.array([180, 255, 255])

mask1 = cv2.inRange(hsv_color, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv_color, lower_red2, upper_red2)
red_mask = cv2.bitwise_or(mask1, mask2)

# Extract only red pixels
red_only = cv2.bitwise_and(color_img, color_img, mask=red_mask)

show_multi(
    [color_img, red_mask, red_only],
    ['Original (Red + Green + Blue circles)', 'Red mask (white = red detected)', 'Only red pixels']
)

print("This technique is used in:")
print("  • Traffic light detection (detect red/green/yellow)")
print("  • Sports ball tracking (bright orange basketball)")
print("  • Medical imaging (detect specific tissue colors)")

---
## 4. Drawing Shapes & Text on Images <a id='4-drawing'></a>

In [ ]:
# ── Drawing primitives ────────────────────────────────────────────────────────

canvas = np.ones((400, 600, 3), dtype=np.uint8) * 240  # light gray background

# cv2.rectangle(img, top-left, bottom-right, BGR-color, thickness)
# thickness=-1 fills the shape
cv2.rectangle(canvas, (20, 20), (180, 120), (0, 120, 255), 3)         # orange border
cv2.rectangle(canvas, (210, 20), (370, 120), (50, 205, 50), -1)       # green filled

# cv2.circle(img, center, radius, BGR-color, thickness)
cv2.circle(canvas, (100, 220), 60, (255, 50, 50), 3)                   # blue border
cv2.circle(canvas, (280, 220), 60, (0, 0, 200), -1)                    # red filled

# cv2.line(img, start, end, BGR-color, thickness)
cv2.line(canvas, (400, 20), (580, 120), (128, 0, 128), 4)              # purple diagonal

# cv2.ellipse(img, center, axes, angle, startAngle, endAngle, color, thickness)
cv2.ellipse(canvas, (480, 220), (80, 50), 30, 0, 360, (0, 200, 200), 3)

# cv2.polylines: draw polygon
pts = np.array([[450, 300], [550, 300], [580, 380], [500, 390], [430, 380]], np.int32)
pts = pts.reshape((-1, 1, 2))
cv2.polylines(canvas, [pts], isClosed=True, color=(200, 100, 0), thickness=2)
cv2.fillPoly(canvas, [pts], (200, 100, 0, 100))

# cv2.putText(img, text, origin, font, scale, color, thickness)
cv2.putText(canvas, 'OpenCV Drawing',
            (20, 370), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)
cv2.putText(canvas, 'font=HERSHEY, scale=0.6, thickness=1',
            (20, 395), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (80, 80, 80), 1)

show(canvas, 'OpenCV Drawing Primitives (rectangle, circle, line, ellipse, polygon, text)',
     figsize=(12, 8))

print("All drawing functions modify the image IN-PLACE (no return value).")
print("Always work on a COPY (img.copy()) to preserve the original.")

---
## 5. Image Transformations <a id='5-transforms'></a>

In [ ]:
# ── Geometric transformations ─────────────────────────────────────────────────

# Create a simple test image with text to show transformations clearly
orig = np.ones((200, 300, 3), dtype=np.uint8) * 255
cv2.rectangle(orig, (20, 20), (130, 90), (200, 50, 50), -1)
cv2.circle(orig, (220, 100), 60, (50, 50, 200), -1)
cv2.putText(orig, 'ORIGINAL', (60, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 2)

h, w = orig.shape[:2]

# 1. Resize
resized_half  = cv2.resize(orig, (w//2, h//2))                                      # half size
resized_ratio = cv2.resize(orig, None, fx=1.5, fy=0.8)                              # scale by factor
resized_inter = cv2.resize(orig, (w*2, h*2), interpolation=cv2.INTER_CUBIC)         # high quality upscale

# 2. Rotation
center = (w // 2, h // 2)
M_rot = cv2.getRotationMatrix2D(center, angle=30, scale=1.0)  # 30 degrees CCW
rotated = cv2.warpAffine(orig, M_rot, (w, h))

# 3. Translation (shift)
M_trans = np.float32([[1, 0, 50], [0, 1, 30]])   # shift right 50, down 30
translated = cv2.warpAffine(orig, M_trans, (w, h))

# 4. Flip
flip_h = cv2.flip(orig, 1)   # 1=horizontal, 0=vertical, -1=both
flip_v = cv2.flip(orig, 0)

# 5. Perspective transform ("birds-eye view")
src_pts = np.float32([[0,0], [w,0], [0,h], [w,h]])
dst_pts = np.float32([[30,0], [w-30,0], [0,h], [w,h]])
M_persp = cv2.getPerspectiveTransform(src_pts, dst_pts)
perspective = cv2.warpPerspective(orig, M_persp, (w, h))

# 6. Crop (just numpy slicing!)
crop = orig[20:120, 50:250]

# Display
row1 = [orig, resized_half, rotated, translated]
title1 = ['Original (300×200)', 'Resize ÷2 (150×100)', 'Rotate 30°', 'Translate +50x+30y']
row2 = [flip_h, flip_v, perspective, np.pad(crop, [(20,0),(50,50),(0,0)], constant_values=200)]
title2 = ['Flip Horizontal', 'Flip Vertical', 'Perspective Transform', 'Crop']

show_multi(row1, title1, figsize=(16, 4))
show_multi(row2, title2, figsize=(16, 4))

print("Interpolation methods for resize:")
print("  INTER_NEAREST  — fastest, blocky (good for masks)")
print("  INTER_LINEAR   — default, bilinear (good balance)")
print("  INTER_CUBIC    — sharper, slower (good for upscaling photos)")
print("  INTER_AREA     — best for shrinking images")

---
## 6. Filtering & Blurring <a id='6-filters'></a>

### The Frosted Glass Analogy

**Blurring = frosted glass** — each pixel becomes an average of its neighbors, smoothing out sharp differences (noise).  
**Sharpening = magnifying glass** — enhances differences between neighboring pixels.  
**Filtering = applying a stencil (kernel)** — slide a small matrix over the image and compute a weighted sum at each position.

In [ ]:
# ── Blur and filter comparison ────────────────────────────────────────────────

# Add noise to test image
noisy = orig.copy().astype(np.float32)
noise = np.random.normal(0, 30, orig.shape)
noisy = np.clip(noisy + noise, 0, 255).astype(np.uint8)

# 1. Box blur (simple average of NxN neighborhood)
box_blur = cv2.blur(noisy, ksize=(9, 9))

# 2. Gaussian blur (weighted average — center pixels weight more)
gaussian = cv2.GaussianBlur(noisy, ksize=(9, 9), sigmaX=0)

# 3. Median blur (replaces pixel with median of neighborhood — great for salt-and-pepper noise)
median = cv2.medianBlur(noisy, ksize=9)

# 4. Bilateral filter (blurs BUT preserves edges — slower but high quality)
bilateral = cv2.bilateralFilter(noisy, d=9, sigmaColor=75, sigmaSpace=75)

# 5. Custom kernel: sharpen
sharpen_kernel = np.array([[ 0, -1,  0],
                            [-1,  5, -1],
                            [ 0, -1,  0]])
sharpened = cv2.filter2D(orig, -1, sharpen_kernel)

# 6. Custom kernel: emboss effect
emboss_kernel = np.array([[-2, -1, 0],
                           [-1,  1, 1],
                           [ 0,  1, 2]])
embossed = cv2.filter2D(orig, -1, emboss_kernel)

row1 = [noisy, box_blur, gaussian, median]
t1   = ['Noisy (Gaussian noise σ=30)', 'Box Blur 9×9', 'Gaussian Blur 9×9', 'Median Blur 9×9']

row2 = [bilateral, sharpened, embossed, orig]
t2   = ['Bilateral Filter (edge-preserving)', 'Sharpened', 'Emboss Effect', 'Original']

show_multi(row1, t1, figsize=(16, 4))
show_multi(row2, t2, figsize=(16, 4))

print("When to use which blur:")
print("  Box/Gaussian:  Pre-processing before edge detection")
print("  Median:        Remove salt-and-pepper noise (individual bright/dark pixels)")
print("  Bilateral:     Denoise photos while keeping edges sharp (e.g., portrait smoothing)")
print("  filter2D:      Any custom convolution kernel")

---
## 7. Edge Detection — Canny Algorithm <a id='7-edges'></a>

### The Sketch Artist Analogy

A sketch artist doesn't draw every pixel — they draw the **outlines** where color changes sharply. **Edge detection** does the same: finds where pixel intensity changes quickly (= boundaries between objects).

**Canny Edge Detection** (1986, John Canny) is the gold standard:
1. Gaussian blur (reduce noise)
2. Sobel gradient (find where intensity changes)
3. Non-maximum suppression (thin edges to 1 pixel wide)
4. Double threshold + hysteresis (keep only real edges)

In [ ]:
# ── Edge detection comparison ─────────────────────────────────────────────────

# Create a more interesting test image for edges
edge_img = np.zeros((300, 400, 3), dtype=np.uint8)
cv2.rectangle(edge_img, (30, 30), (150, 150), (180, 120, 60), -1)
cv2.circle(edge_img,    (300, 150), 90, (80, 160, 220), -1)
cv2.ellipse(edge_img,   (200, 250), (120, 40), 0, 0, 360, (120, 200, 100), -1)
edge_img = cv2.GaussianBlur(edge_img, (3, 3), 0)  # slight blur for realism

gray_edge = cv2.cvtColor(edge_img, cv2.COLOR_BGR2GRAY)

# 1. Sobel (computes gradient in X or Y direction)
sobelx = cv2.Sobel(gray_edge, cv2.CV_64F, 1, 0, ksize=3)  # horizontal edges
sobely = cv2.Sobel(gray_edge, cv2.CV_64F, 0, 1, ksize=3)  # vertical edges
sobel_combined = cv2.magnitude(sobelx, sobely).astype(np.uint8)

# 2. Laplacian (detects edges in all directions at once)
laplacian = cv2.Laplacian(gray_edge, cv2.CV_64F)
laplacian = np.absolute(laplacian).astype(np.uint8)

# 3. Canny (best quality — threshold1, threshold2)
# threshold1: below → discard. threshold2: above → definite edge. Between → only if connected to definite edge
canny_tight  = cv2.Canny(gray_edge, threshold1=100, threshold2=200)  # fewer edges
canny_loose  = cv2.Canny(gray_edge, threshold1=30,  threshold2=100)  # more edges

# Canny on blurred image (reduce noise first)
blurred_gray = cv2.GaussianBlur(gray_edge, (5, 5), 0)
canny_smooth = cv2.Canny(blurred_gray, 50, 150)

row1 = [edge_img, gray_edge, sobel_combined, laplacian]
t1   = ['Original', 'Grayscale', 'Sobel (magnitude)', 'Laplacian']
row2 = [canny_tight, canny_loose, canny_smooth, cv2.bitwise_not(canny_smooth)]
t2   = ['Canny (100,200) tight', 'Canny (30,100) loose', 'Canny on blurred', 'Inverted Canny (sketch look)']

show_multi(row1, t1, figsize=(16, 4))
show_multi(row2, t2, figsize=(16, 4))

print("Canny threshold tuning:")
print("  Low thresholds → more edges (includes noise)")
print("  High thresholds → fewer, cleaner edges")
print("  Rule of thumb: threshold2 ≈ 2–3 × threshold1")
print("  Pre-blur with Gaussian to reduce noise before Canny")

---
## 8. Contours — Finding Object Boundaries <a id='8-contours'></a>

### The Cookie Cutter Analogy

After edge detection you have lines on a page. **Contours** connect those lines into closed shapes — like a cookie cutter outline. From contours you can measure area, perimeter, bounding box, shape classification, and more.

In [ ]:
# ── Find and analyze contours ─────────────────────────────────────────────────

# Create image with distinct shapes
shapes_img = np.zeros((350, 500, 3), dtype=np.uint8)
cv2.rectangle(shapes_img, (30,  30), (150, 130), (255, 255, 255), -1)  # rectangle
cv2.circle(shapes_img,   (280, 80),  70, (255, 255, 255), -1)          # circle
pts_tri = np.array([[430,30],[480,130],[380,130]], np.int32)
cv2.fillPoly(shapes_img, [pts_tri], (255, 255, 255))                    # triangle
cv2.ellipse(shapes_img, (120, 260), (80, 50), 0, 0, 360, (255,255,255), -1)  # ellipse
cv2.fillPoly(shapes_img,
    [np.array([[300,200],[380,200],[420,280],[340,320],[270,280]], np.int32)],
    (255,255,255))                                                       # pentagon

# Convert to grayscale and find contours
gray_shapes = cv2.cvtColor(shapes_img, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray_shapes, 127, 255, cv2.THRESH_BINARY)

# findContours: returns list of contour points
# RETR_EXTERNAL: only outermost contours
# CHAIN_APPROX_SIMPLE: compress horizontal/vertical/diagonal segments
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"Found {len(contours)} contours (shapes)")

annotated = shapes_img.copy()
shape_names = []

for i, cnt in enumerate(contours):
    area      = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, closed=True)

    # Approximate contour to polygon
    epsilon = 0.03 * perimeter
    approx  = cv2.approxPolyDP(cnt, epsilon, closed=True)
    n_vertices = len(approx)

    # Classify shape by vertex count
    circularity = (4 * np.pi * area) / (perimeter ** 2 + 1e-6)
    if circularity > 0.85:
        shape = 'Circle'
    elif n_vertices == 3:
        shape = 'Triangle'
    elif n_vertices == 4:
        x, y, w, h = cv2.boundingRect(approx)
        aspect = w / float(h)
        shape = 'Square' if 0.9 <= aspect <= 1.1 else 'Rectangle'
    elif n_vertices == 5:
        shape = 'Pentagon'
    else:
        shape = f'Polygon({n_vertices})'

    # Draw contour and label
    color = (0, int(255 * i / len(contours)), 255)
    cv2.drawContours(annotated, [cnt], -1, color, 2)

    M = cv2.moments(cnt)
    if M['m00'] > 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        cv2.putText(annotated, shape, (cx-30, cy),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
        cv2.putText(annotated, f'A={area:.0f}', (cx-30, cy+16),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.38, (180, 255, 180), 1)

    shape_names.append(shape)
    print(f"  Contour {i+1}: {shape:12s} | area={area:6.0f} | vertices={n_vertices} | circularity={circularity:.2f}")

show_multi([shapes_img, annotated],
           ['Original shapes', 'Detected contours with classification'],
           figsize=(14, 6))

---
## 9. Morphological Operations <a id='9-morphology'></a>

Morphological operations are simple rules for **growing** or **shrinking** white regions in binary images — like expanding or eroding a sand castle:

| Operation | Effect | Use Case |
|---|---|---|
| **Erosion** | Shrinks white regions (removes small noise) | Remove tiny blobs |
| **Dilation** | Grows white regions (fills small holes) | Connect broken edges |
| **Opening** (erode → dilate) | Removes small white noise | Clean up noisy masks |
| **Closing** (dilate → erode) | Fills small black holes | Fill gaps in objects |

In [ ]:
# ── Morphological operations demo ─────────────────────────────────────────────

# Create noisy binary image
morph_base = np.zeros((200, 300), dtype=np.uint8)
cv2.rectangle(morph_base, (30, 30), (200, 170), 255, -1)
# Add noise: random pixels
noise_mask = np.random.rand(*morph_base.shape) > 0.95
morph_base[noise_mask] = 255 - morph_base[noise_mask]  # flip noisy pixels

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))

erosion  = cv2.erode(morph_base, kernel, iterations=1)
dilation = cv2.dilate(morph_base, kernel, iterations=1)
opening  = cv2.morphologyEx(morph_base, cv2.MORPH_OPEN,  kernel)  # erode → dilate
closing  = cv2.morphologyEx(morph_base, cv2.MORPH_CLOSE, kernel)  # dilate → erode
gradient = cv2.morphologyEx(morph_base, cv2.MORPH_GRADIENT, kernel)  # edge outline
tophat   = cv2.morphologyEx(morph_base, cv2.MORPH_TOPHAT, kernel)    # bright spots on dark

row1 = [morph_base, erosion, dilation, opening]
t1   = ['Noisy Binary', 'Erosion (shrinks)', 'Dilation (grows)', 'Opening (remove noise)']
row2 = [closing, gradient, tophat, np.zeros_like(morph_base)]
t2   = ['Closing (fill holes)', 'Gradient (outline)', 'Top Hat (bright spots)', '']

show_multi(row1, t1, figsize=(16, 4))
show_multi(row2[:3], t2[:3], figsize=(12, 4))

print("\nKernel shapes available:")
print("  cv2.MORPH_RECT    — rectangular kernel (most common)")
print("  cv2.MORPH_ELLIPSE — elliptical kernel (smoother)")
print("  cv2.MORPH_CROSS   — cross-shaped kernel")

---
## 10. Histograms & Histogram Equalization <a id='10-histograms'></a>

In [ ]:
# ── Histogram analysis and equalization ──────────────────────────────────────

# Create a dark, low-contrast image
dark_img = np.zeros((200, 300, 3), dtype=np.uint8)
cv2.rectangle(dark_img, (20, 20), (150, 120), (50, 60, 70), -1)
cv2.circle(dark_img, (230, 100), 70, (40, 80, 50), -1)
cv2.putText(dark_img, 'LOW CONTRAST', (10, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (60,60,60), 1)

dark_gray = cv2.cvtColor(dark_img, cv2.COLOR_BGR2GRAY)

# Histogram equalization (spreads pixel values across 0-255)
equalized = cv2.equalizeHist(dark_gray)

# CLAHE: Contrast Limited Adaptive Histogram Equalization (better for local regions)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
clahe_img = clahe.apply(dark_gray)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

imgs   = [dark_gray, equalized, clahe_img]
titles = ['Original (low contrast)', 'Histogram Equalized', 'CLAHE (adaptive)']

for col in range(3):
    # Show image
    axes[0, col].imshow(imgs[col], cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(titles[col], fontsize=11)
    axes[0, col].axis('off')

    # Show histogram
    hist = cv2.calcHist([imgs[col]], [0], None, [256], [0, 256])
    axes[1, col].fill_between(range(256), hist.flatten(), alpha=0.7, color='steelblue')
    axes[1, col].set_title(f'Histogram of {titles[col]}')
    axes[1, col].set_xlabel('Pixel intensity (0=black, 255=white)')
    axes[1, col].set_ylabel('Number of pixels')
    axes[1, col].set_xlim([0, 256])
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle('Histogram Equalization: Spreading Pixel Values for Better Contrast', fontsize=13)
plt.tight_layout()
plt.show()

print("Use cases:")
print("  • Medical imaging: X-ray / MRI contrast enhancement")
print("  • Surveillance: improve visibility in dark footage")
print("  • CLAHE preferred for natural images (avoids over-amplifying noise)")

---
## 11. Template Matching <a id='11-template'></a>

In [ ]:
# ── Template matching: find a small image inside a larger one ─────────────────

# Create a scene with multiple objects
scene = np.ones((300, 500, 3), dtype=np.uint8) * 200

# Place a "target" shape (red square) and distractors
cv2.rectangle(scene, (80, 60), (130, 110), (0, 0, 180), -1)   # target 1
cv2.circle(scene, (250, 150), 35, (0, 180, 0), -1)             # distractor
cv2.rectangle(scene, (360, 60), (410, 110), (0, 0, 180), -1)  # target 2 (same shape)
cv2.rectangle(scene, (180, 200), (230, 250), (0, 0, 180), -1) # target 3
cv2.ellipse(scene, (430, 200), (50, 30), 0, 0, 360, (0, 120, 120), -1)  # distractor

# Extract template: the red square (30×30 region)
template = scene[60:110, 80:130].copy()
t_h, t_w = template.shape[:2]

# Convert to grayscale for matching
scene_gray    = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
template_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

# Apply template matching
# TM_CCOEFF_NORMED: normalized cross-correlation (best for general use, range [-1,1])
result = cv2.matchTemplate(scene_gray, template_gray, cv2.TM_CCOEFF_NORMED)

# Find locations where result > threshold
threshold = 0.85
locations = np.where(result >= threshold)

annotated_scene = scene.copy()
matches = 0
for pt in zip(*locations[::-1]):  # (x, y) from (row, col)
    cv2.rectangle(annotated_scene, pt, (pt[0]+t_w, pt[1]+t_h), (0, 255, 0), 3)
    matches += 1

print(f"Template matching found {matches} locations (threshold={threshold})")

# Show result heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(template, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Template ({t_w}×{t_h} px)'); axes[0].axis('off')

im = axes[1].imshow(result, cmap='hot', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1])
axes[1].set_title('Match Score Heatmap\n(bright = good match)')

axes[2].imshow(cv2.cvtColor(annotated_scene, cv2.COLOR_BGR2RGB))
axes[2].set_title(f'Detected matches (score ≥ {threshold})')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\nTemplate matching methods:")
print("  TM_SQDIFF         — sum of squared differences (lower=better)")
print("  TM_SQDIFF_NORMED  — normalized, range [0,1] (lower=better)")
print("  TM_CCOEFF         — cross-correlation")
print("  TM_CCOEFF_NORMED  — normalized [-1,1] (higher=better) ← USE THIS")
print("\nLimitation: fails with rotation/scale changes → use SIFT/ORB for that")

---
## 12. Video Processing Basics <a id='12-video'></a>

In [ ]:
# ── Synthetic video processing demo ──────────────────────────────────────────
# (Real video needs cv2.VideoCapture; here we simulate frame-by-frame processing)

print("Video in OpenCV = sequence of frames processed in a loop")
print()
print("=== Real camera / video file code pattern ===")
print("""
cap = cv2.VideoCapture(0)          # 0=webcam, or 'video.mp4' for file

# Get video properties
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Resolution: {width}x{height} @ {fps} FPS")

# Save output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()     # ret=False when video ends
    if not ret:
        break

    # --- Process each frame here ---
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    # --------------------------------

    cv2.imshow('Video', frame)
    out.write(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):   # press 'q' to quit
        break

cap.release()
out.release()
cv2.destroyAllWindows()
""")

# ── Simulate processing 5 frames ─────────────────────────────────────────────
print("\n=== Simulating frame processing ===")

n_frames = 5
processed_frames = []
titles = []

for i in range(n_frames):
    # Simulate a moving ball
    frame = np.zeros((150, 200, 3), dtype=np.uint8)
    x = 30 + i * 35
    cv2.circle(frame, (x, 75), 25, (0, 120, 255), -1)
    cv2.putText(frame, f'Frame {i}', (5, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)

    # Processing: Canny edges
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 30, 100)
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # Side by side
    combined = np.hstack([frame, edges_bgr])
    processed_frames.append(combined)
    titles.append(f'Frame {i} | original & edges')

show_multi(processed_frames, titles, figsize=(20, 3))
print("\nFPS considerations:")
print("  • 30 FPS → 33ms per frame budget for all processing")
print("  • Profile with time.perf_counter() to find bottlenecks")
print("  • Use cv2.resize() to smaller resolution before heavy processing")

---
## 13. Mini Project — Coin Counter <a id='13-mini-project'></a>

### What We're Building

A pipeline that automatically counts coins in an image and estimates their total value — using only classical OpenCV (no deep learning needed).

**Real-world equivalent:** Vending machines, automated cash counting systems, manufacturing quality control.

In [ ]:
# ── Generate synthetic coin image ─────────────────────────────────────────────

np.random.seed(7)

# Canvas
coin_canvas = np.ones((400, 600, 3), dtype=np.uint8) * 50  # dark background

# Add some texture to background
noise = np.random.randint(0, 20, coin_canvas.shape, dtype=np.uint8)
coin_canvas = cv2.add(coin_canvas, noise)

# Define coin sizes (radius) and values
COIN_TYPES = [
    {'value': 1,   'radius': 30, 'color': (120, 170, 200), 'name': '$1'},
    {'value': 0.5, 'radius': 25, 'color': (160, 190, 210), 'name': '50c'},
    {'value': 0.25,'radius': 22, 'color': (170, 170, 180), 'name': '25c'},
    {'value': 0.10,'radius': 18, 'color': (180, 160, 140), 'name': '10c'},
    {'value': 0.05,'radius': 15, 'color': (140, 120, 110), 'name': '5c'},
]

coin_positions = []
coin_values    = []
ground_truth_count = 0

# Place coins randomly (non-overlapping)
def coins_overlap(cx, cy, r, placed, margin=5):
    for px, py, pr in placed:
        if np.sqrt((cx-px)**2 + (cy-py)**2) < r + pr + margin:
            return True
    return False

placed = []
# Distribute: 3×$1, 4×50c, 5×25c, 4×10c, 4×5c
coin_plan = [0]*3 + [1]*4 + [2]*5 + [3]*4 + [4]*4
np.random.shuffle(coin_plan)

for coin_type_idx in coin_plan:
    ct = COIN_TYPES[coin_type_idx]
    r  = ct['radius']
    for _ in range(100):  # try up to 100 random placements
        cx = np.random.randint(r+10, 590-r)
        cy = np.random.randint(r+10, 390-r)
        if not coins_overlap(cx, cy, r, placed):
            placed.append((cx, cy, r))
            coin_positions.append((cx, cy, r, ct['value'], ct['name']))
            ground_truth_count += 1
            # Draw coin
            cv2.circle(coin_canvas, (cx, cy), r, ct['color'], -1)
            # Shiny edge
            cv2.circle(coin_canvas, (cx, cy), r, (220, 220, 220), 2)
            # Coin ridge
            cv2.circle(coin_canvas, (cx, cy), r-4, (ct['color'][0]-20, ct['color'][1]-20, ct['color'][2]-20), 1)
            coin_values.append(ct['value'])
            break

total_value = sum(coin_values)
print(f"Synthetic coin image: {ground_truth_count} coins placed, total = ${total_value:.2f}")
show(coin_canvas, f'Synthetic coin image: {ground_truth_count} coins, total=${total_value:.2f}',
     figsize=(12, 8))

In [ ]:
# ── Coin counting pipeline ────────────────────────────────────────────────────

def count_coins(img, debug=True):
    """
    Count coins in image using Hough Circle Transform.
    
    Pipeline:
        1. Grayscale conversion
        2. Gaussian blur (reduce noise before circle detection)
        3. HoughCircles (detect circles at multiple radii)
        4. Draw detected circles
        5. Classify coin size by radius
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Blur before Hough (essential — reduces false positives)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)

    # HoughCircles parameters:
    # dp=1: inverse accumulator resolution
    # minDist: minimum distance between circle centers (prevents multiple detections of same coin)
    # param1: Canny upper threshold
    # param2: accumulator threshold (lower → more (false) circles)
    # minRadius, maxRadius: search range
    circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=40,
        param1=50,
        param2=30,
        minRadius=10,
        maxRadius=40
    )

    result = img.copy()
    detected_coins = []

    if circles is not None:
        circles = np.round(circles[0, :]).astype(int)   # shape: (N, 3) — x, y, r

        for x, y, r in circles:
            # Classify by radius (tuned to our synthetic coin sizes)
            if r >= 27:
                value, name = 1.00, '$1'
                color = (0, 255, 0)
            elif r >= 22:
                value, name = 0.50, '50c'
                color = (0, 200, 255)
            elif r >= 18:
                value, name = 0.25, '25c'
                color = (255, 200, 0)
            elif r >= 14:
                value, name = 0.10, '10c'
                color = (200, 100, 255)
            else:
                value, name = 0.05, '5c'
                color = (100, 200, 255)

            # Draw circle outline and center
            cv2.circle(result, (x, y), r, color, 2)
            cv2.circle(result, (x, y), 2, (255, 255, 255), -1)

            # Label
            cv2.putText(result, name, (x-15, y+5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

            detected_coins.append({'x': x, 'y': y, 'radius': r,
                                   'value': value, 'name': name})

    total = sum(c['value'] for c in detected_coins)

    # Summary box
    cv2.rectangle(result, (0, 0), (220, 60), (0, 0, 0), -1)
    cv2.putText(result, f'Detected: {len(detected_coins)} coins',
               (5, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    cv2.putText(result, f'Total: ${total:.2f}',
               (5, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 200), 2)

    return result, detected_coins, total


result_img, detected, total_detected = count_coins(coin_canvas)

show_multi([coin_canvas, result_img],
           ['Original Coin Image', f'Detected: {len(detected)} coins | Total: ${total_detected:.2f}'],
           figsize=(16, 7))

print(f"\n{'═'*45}")
print(f"COIN COUNTING RESULTS")
print(f"{'─'*45}")
print(f"Ground truth:  {ground_truth_count} coins, ${total_value:.2f}")
print(f"Detected:      {len(detected)} coins, ${total_detected:.2f}")
print(f"Accuracy:      {len(detected)/ground_truth_count*100:.1f}% detection rate")

from collections import Counter
by_type = Counter(c['name'] for c in detected)
print(f"\nBreakdown by coin type:")
for name, count in sorted(by_type.items()):
    val = {'$1':1.0, '50c':0.5, '25c':0.25, '10c':0.10, '5c':0.05}[name]
    print(f"  {name}: {count} coins × {val:.2f} = ${count*val:.2f}")

---
## 14. Common Pitfalls <a id='14-pitfalls'></a>

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║              OPENCV — COMMON PITFALLS                        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. WRONG: Displaying with matplotlib without BGR→RGB        ║
║     plt.imshow(cv2.imread('photo.jpg'))  ← COLORS ARE WRONG  ║
║  RIGHT:                                                      ║
║     img = cv2.imread('photo.jpg')                            ║
║     plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  ✓      ║
║                                                              ║
║  2. WRONG: Modifying original image with drawing functions   ║
║     cv2.rectangle(img, ...)   ← modifies img IN-PLACE!       ║
║  RIGHT: always copy first                                    ║
║     annotated = img.copy()                                   ║
║     cv2.rectangle(annotated, ...)  ✓                         ║
║                                                              ║
║  3. WRONG: Passing wrong dtype to functions                  ║
║     img_float = img.astype(np.float32)                       ║
║     gray = cv2.cvtColor(img_float, cv2.COLOR_BGR2GRAY)       ║
║     cv2.Canny(gray, 50, 150)  ← Canny needs uint8!           ║
║  RIGHT: cv2.Canny(gray.astype(np.uint8), 50, 150)            ║
║                                                              ║
║  4. WRONG: Hardcoded thresholds for all images               ║
║     cv2.Canny(gray, 50, 150)  ← works for one image only     ║
║  RIGHT: Use Otsu's threshold, or compute from histogram      ║
║     _, thresh = cv2.threshold(gray, 0, 255,                  ║
║                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)       ║
║                                                              ║
║  5. WRONG: Shape confusion — img.shape vs img.shape[:2]      ║
║     h, w = img.shape   ← FAILS for color image (3-tuple)     ║
║  RIGHT: h, w = img.shape[:2]  ← works for both gray & color  ║
║                                                              ║
║  6. WRONG: Not releasing video capture                       ║
║     # Forgetting cap.release() → webcam stays locked!        ║
║  RIGHT: Use try/finally or context manager                   ║
║     try:                                                     ║
║         cap = cv2.VideoCapture(0)                            ║
║         ...                                                  ║
║     finally:                                                 ║
║         cap.release()                                        ║
║         cv2.destroyAllWindows()                              ║
║                                                              ║
║  7. WRONG: Using BGR colors in cvtColor output as if RGB     ║
║     hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)               ║
║     # H range in OpenCV HSV is 0-179, NOT 0-360!             ║
║     # Divide H by 2 when comparing with standard HSV values  ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## 15. Interview Q&A <a id='15-interview'></a>

In [ ]:
qa = [
    ("Why does OpenCV use BGR instead of RGB?",
     "Historical reason: when OpenCV was first developed (early 2000s), BGR was the standard "
     "for many camera manufacturers and image file formats in use at the time. Microsoft's "
     "bitmap format (.bmp) used BGR, and OpenCV was initially developed for Windows. "
     "The convention stuck. Always convert with cv2.cvtColor(img, cv2.COLOR_BGR2RGB) "
     "before using matplotlib or PIL/Pillow."),

    ("How does the Canny edge detector work?",
     "5-step algorithm: (1) Gaussian blur — reduce noise. (2) Sobel gradient — compute "
     "intensity gradient magnitude and direction at each pixel. (3) Non-maximum suppression — "
     "thin edges to 1 pixel by keeping only local maxima along gradient direction. "
     "(4) Double threshold — classify pixels as strong (>high), weak (between thresholds), "
     "or suppressed (<low). (5) Hysteresis edge tracking — keep weak edges only if connected "
     "to a strong edge. Result: clean, thin, connected edges."),

    ("When would you use HSV color space instead of BGR?",
     "HSV is ideal for COLOR-BASED detection/segmentation. In BGR, detecting 'red objects' "
     "is hard because lighting changes all three values. In HSV, the hue (color type) is "
     "separated from saturation (color purity) and value (brightness). A red apple in "
     "bright sunlight and in shade have very different BGR values but similar hue values. "
     "Use cv2.inRange() on HSV for robust color masking."),

    ("What is the difference between erosion and dilation?",
     "Both operate on binary images with a structuring element (kernel). "
     "Erosion: a pixel is white only if ALL pixels under the kernel are white. "
     "→ Shrinks white regions, removes small white noise. "
     "Dilation: a pixel is white if ANY pixel under the kernel is white. "
     "→ Grows white regions, fills small holes. "
     "Opening (erode then dilate) removes small white blobs. "
     "Closing (dilate then erode) fills small dark holes inside white regions."),

    ("How does template matching work and what are its limitations?",
     "Template matching slides a template image across a larger image and computes a "
     "similarity score at each position. TM_CCOEFF_NORMED gives normalized cross-correlation "
     "[-1,1]; find the peak for the best match. "
     "Limitations: (1) fails with rotation or scale changes — the template must be "
     "same size and orientation; (2) sensitive to lighting changes; (3) O(W×H×t_W×t_H) "
     "complexity — slow for large images or many templates. "
     "For rotation/scale invariance, use feature-based matching: SIFT, ORB, SURF."),

    ("What is Hough Circle Transform and what does param2 control?",
     "Hough Circle Transform detects circles by voting in an accumulator space (center x, center y, radius). "
     "Each edge pixel votes for all circles that could have produced it. "
     "param1: Canny upper threshold used internally. "
     "param2: accumulator threshold — higher means fewer but more confident circles detected. "
     "Lower param2 → more circles found (including false positives). "
     "minDist controls minimum distance between circle centers to avoid multiple detections. "
     "Always Gaussian-blur before Hough — circles are sensitive to noise."),

    ("What does cv2.findContours return and what are the flags?",
     "Returns (contours, hierarchy). contours = list of NumPy arrays of shape (N,1,2). "
     "RETR modes: RETR_EXTERNAL (outermost only), RETR_LIST (all, flat), "
     "RETR_TREE (full hierarchy with parent-child relationships). "
     "CHAIN_APPROX modes: CHAIN_APPROX_NONE (all points), "
     "CHAIN_APPROX_SIMPLE (compress straight segments → fewer points). "
     "Use cv2.contourArea(), cv2.arcLength(), cv2.boundingRect(), "
     "cv2.minEnclosingCircle() to analyze contours."),
]

print("=" * 70)
print("  INTERVIEW Q&A — OPENCV")
print("=" * 70)
for i, (q, a) in enumerate(qa, 1):
    print(f"\nQ{i}: {q}")
    print(f"\nA{i}: {a}")
    print("\n" + "─" * 70)

---
## 16. Resources <a id='16-resources'></a>

### Official Documentation
- **OpenCV docs**: https://docs.opencv.org/4.x/
- **OpenCV Python tutorials**: https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html

### Books
- **Learning OpenCV 4** (Kaehler & Bradski) — the definitive reference
- **Programming Computer Vision with Python** (Jan Erik Solem) — free PDF available

### Video Tutorials
- **OpenCV full course** (freeCodeCamp): https://youtu.be/oXlwWbU8l2o
- **Computer Vision crash course**: https://youtu.be/WQeoO7MI0Bs
- **OpenCV Python tutorials playlist**: https://www.youtube.com/playlist?list=PLS1QulWo1RIa7D1O6skqDQ-JZ1GGHKK-K

### Advanced Topics (after OpenCV mastery)
- **Feature detection**: SIFT, ORB, SURF — `cv2.SIFT_create()`
- **Optical flow**: `cv2.calcOpticalFlowPyrLK()` — track moving objects
- **Camera calibration**: `cv2.calibrateCamera()` — remove lens distortion
- **Stereo vision**: `cv2.StereoBM_create()` — depth from two cameras
- **Background subtraction**: `cv2.createBackgroundSubtractorMOG2()`

## Interview Questions & Answers

---

**Q1: How does OpenCV represent images and why does it use BGR instead of RGB?**

A: OpenCV stores images as NumPy arrays of shape `(H, W, C)` where C=3 for colour (BGR) or C=1 for grayscale. BGR (Blue-Green-Red) is a historical artefact from the early days of video capture hardware. Always convert with `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` before using matplotlib (which expects RGB) or before feeding to neural networks trained on RGB data. Forgetting this makes red appear blue — a very common beginner bug.

---

**Q2: What is the difference between erosion and dilation in morphological operations?**

A: Both operate by sliding a **structuring element** (kernel) over binary images. **Erosion** keeps a pixel white only if ALL kernel pixels land on white — shrinks white regions, removes noise/thin protrusions. **Dilation** makes a pixel white if ANY kernel pixel lands on white — grows white regions, fills holes. Common combinations: **Opening** (erosion → dilation) removes small noise. **Closing** (dilation → erosion) fills small holes. Used in document scanning, fingerprint enhancement, and preprocessing for OCR.

---

**Q3: What is the Canny edge detector and what are its four stages?**

A: Canny (1986) is still the standard edge detector:
1. **Gaussian blur** — reduce noise (otherwise noise = false edges)
2. **Sobel gradient** — compute edge strength and direction
3. **Non-maximum suppression** — thin edges to 1 pixel wide
4. **Double threshold + hysteresis** — strong edges kept, weak edges kept only if connected to strong ones
The two threshold values `(low, high)` in `cv2.Canny(img, low, high)` control this. Rule of thumb: `high = 3 × low`.

---

**Q4: How does template matching work and when does it fail?**

A: `cv2.matchTemplate()` slides a small template image over the source, computing a similarity score at each position. `cv2.TM_CCOEFF_NORMED` returns 1.0 for perfect match. Fails when: (1) **scale changes** — the same object appears larger/smaller; (2) **rotation** — template is fixed orientation; (3) **lighting/colour changes** — normalised correlation helps somewhat; (4) **occlusion** — template partially hidden. For robust matching, use feature-based methods (ORB, SIFT) which are scale- and rotation-invariant.

---

**Q5: What is optical flow and what is it used for?**

A: Optical flow estimates the per-pixel motion vector between two consecutive video frames. **Lucas-Kanade** (`cv2.calcOpticalFlowPyrLK`) tracks sparse feature points — fast, good for tracking specific objects. **Farneback** (`cv2.calcOpticalFlowFarneback`) computes dense flow (every pixel) — slower but gives full motion field. Applications: action recognition, video stabilisation, autonomous driving (ego-motion estimation), sports analytics (ball/player tracking).

---

**Q6: When would you use OpenCV vs a deep learning model for computer vision?**

A: **Use OpenCV (classical CV)** when: you need real-time speed on CPU; the task is geometric (rotation, homography, edge detection); you have no labelled training data; interpretability is required (medical, legal). **Use deep learning** when: appearance varies greatly (lighting, texture, deformation); accuracy matters more than speed; you have 1000+ labelled examples; the task is semantic ("is this a cat?"). In practice, hybrid: OpenCV for preprocessing + DL for inference is the most common production pattern.

## Recommended Resources

| Resource | Link |
|---|---|
| OpenCV Docs | https://docs.opencv.org/ |
| OpenCV Python Tutorials | https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html |
| Practical Python & OpenCV | https://pyimagesearch.com/ |
| Canny Paper | https://ieeexplore.ieee.org/document/4767851 |
| OpenCV GitHub | https://github.com/opencv/opencv |


---
## 17. Summary & What's Next <a id='17-summary'></a>

### What You Learned

| Concept | Key Takeaway |
|---|---|
| Image representation | NumPy array (H, W, 3), uint8, **BGR** not RGB |
| Color spaces | BGR → HSV for color detection; grayscale for processing |
| Geometric transforms | resize, rotate, translate, flip, perspective warp |
| Filtering | Gaussian/median blur to denoise; bilateral to preserve edges |
| Edge detection | Canny = blur + gradient + NMS + hysteresis |
| Contours | findContours → analyze shape, area, perimeter |
| Morphology | Erode (shrink) / Dilate (grow) / Open / Close |
| Histograms | Equalization/CLAHE to fix low-contrast images |
| Template matching | Slide template over image; `TM_CCOEFF_NORMED` |
| Video | Frame loop: cap.read() → process → display → waitKey |

### What's Next

| Notebook | Topic |
|---|---|
| `Torchvision` | Deep learning for images: CNNs, transfer learning, augmentation |
| `YOLO_Ultralytics` | Real-time object detection and tracking |
| `Detectron2` | Instance segmentation and keypoint detection |

**OpenCV is your toolkit for image manipulation. Torchvision + YOLO is where you add intelligence to recognize *what* is in the image!**